# ⚙️ Notebook 00: Project Setup, Hardware Telemetry & Environment Configuration
### Capstone: CLIP vs. V-JEPA for CCTV Event Retrieval, Summarization, LLM Reporting & Hallucination Analysis
---
**Objective:** Establish the foundational infrastructure, audit GPU acceleration hardware on the **NVIDIA RTX 4000 Ada Generation (20GB)**, configure reproducible software environments, validate dataset paths across all **4 active CCTV datasets (5,153 real videos)**, and verify global configuration parameters.

#### Research Framework Overview:
This research investigates fine-grained spatiotemporal CCTV surveillance event retrieval, forensic video summarization, LLM-based incident report generation, and automated claim-level hallucination detection.
- **Visual Backbones:** CLIP (Contrastive Language-Image Pretraining) vs. V-JEPA (Video Joint Embedding Predictive Architecture).
- **Core Hypotheses:**
  - **H1 (Retrieval Quality):** V-JEPA + Spatiotemporal Cross-Modal Predictor achieves superior spatiotemporal retrieval over static frame pooling.
  - **H2 (Cross-Camera Generalization):** Spatiotemporal joint embeddings reduce cross-viewpoint degradation across Aerial, Ground, and Rooftop camera viewpoints.
  - **H3 (Report Faithfulness):** Video-grounded RAG reporting eliminates visual hallucination compared to ungrounded LLM narrative generation.


## Step 1: System Imports & Root Directory Setup
Configure workspace paths to allow bidirectional importing between the notebook environment and the core `implementation` codebase.


In [ ]:
import sys
import os
import time
import json
import psutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

# Resolve project repository root
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "CLIP_vs_VJEPA_CCTV_Event_Retrieval_and_Hallucination_Analysis" else NOTEBOOK_DIR

sys.path.append(str(REPO_ROOT / "notebooks"))
sys.path.append(str(REPO_ROOT / "implementation"))
sys.path.append(str(REPO_ROOT))

import notebook_utils as nu

print(f"[Setup] Working Directory: {NOTEBOOK_DIR}")
print(f"[Setup] Repository Root:   {REPO_ROOT}")
print(f"[Setup] PyTorch Version:    {torch.__version__}")
print(f"[Setup] CUDA Available:     {torch.cuda.is_available()}")


## Step 2: GPU Hardware Telemetry & Compute Capabilities
Detailed audit of NVIDIA RTX 4000 Ada Generation GPU resources, compute architecture, VRAM capacity, and CUDA execution streams.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Active Compute Device: {device.upper()}")

if device == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("=" * 60)
    print(f"GPU Model:                {props.name}")
    print(f"Compute Capability:       SM {props.major}.{props.minor}")
    print(f"Total Dedicated VRAM:     {props.total_memory / (1024**3):.2f} GB")
    print(f"Streaming Multiprocessors:{props.multi_processor_count}")
    print(f"PyTorch CUDA Arch:        {torch.cuda.get_arch_list()}")
    print("=" * 60)
    
    # Simple tensor core warmup
    x = torch.randn(2048, 2048, device=device, dtype=torch.float16)
    t0 = time.time()
    for _ in range(50):
        y = torch.matmul(x, x)
    torch.cuda.synchronize()
    print(f"GPU Warmup FP16 MatMul (50x 2048x2048): {(time.time() - t0)*1000:.2f} ms")
else:
    print("WARNING: CUDA acceleration unavailable! Running on CPU fallback.")


## Step 3: Software Dependencies & Libraries Audit
Verify the versions of key scientific computing, deep learning, vector search, and computer vision libraries.


In [ ]:
import faiss
import cv2
import transformers

dep_data = [
    {"Library": "PyTorch", "Installed Version": torch.__version__, "Required Min": "2.0.0"},
    {"Library": "Torchvision", "Installed Version": getattr(sys.modules.get('torchvision'), '__version__', 'Available'), "Required Min": "0.15.0"},
    {"Library": "Transformers (HuggingFace)", "Installed Version": transformers.__version__, "Required Min": "4.30.0"},
    {"Library": "FAISS", "Installed Version": faiss.__version__, "Required Min": "1.7.4"},
    {"Library": "OpenCV (cv2)", "Installed Version": cv2.__version__, "Required Min": "4.8.0"},
    {"Library": "Pandas", "Installed Version": pd.__version__, "Required Min": "2.0.0"},
    {"Library": "NumPy", "Installed Version": np.__version__, "Required Min": "1.24.0"},
]

df_deps = pd.DataFrame(dep_data)
display(df_deps)


## Step 4: Master Configuration & Directory Structure
Load master experiment parameters from `config.yaml` and create standard project directories.


In [ ]:
# Create required directory hierarchy
dirs_to_create = [
    REPO_ROOT / "implementation" / "results" / "embeddings",
    REPO_ROOT / "implementation" / "results" / "indices",
    REPO_ROOT / "implementation" / "results" / "reports",
    REPO_ROOT / "notebooks" / "outputs" / "figures",
    REPO_ROOT / "notebooks" / "outputs" / "tables",
    REPO_ROOT / "notebooks" / "outputs" / "checkpoints",
    REPO_ROOT / "notebooks" / "outputs" / "trace_logs",
]

for d in dirs_to_create:
    d.mkdir(parents=True, exist_ok=True)
    print(f"[Dir Verified] {d.relative_to(REPO_ROOT)}")

# Load master config
config_path = REPO_ROOT / "implementation" / "config" / "config.yaml"
if config_path.exists():
    import yaml
    with open(config_path, "r", encoding="utf-8") as f:
        master_config = yaml.safe_load(f)
    print(f"\nSuccessfully loaded master configuration from: {config_path.name}")
    print(f"Experiment Name: {master_config.get('experiment', {}).get('name')}")
else:
    print(f"Config path not found at {config_path}")


## Step 5: Multi-Dataset Path & Integrity Verification
Verify accessibility of the 4 real-world surveillance datasets across the local workstation.


In [ ]:
dataset_paths = {
    "RLVS (Violence)": Path(r"C:\Users\HPCLAB\Desktop\VLJEPA\capstone-data-root\real life voilence data\Real Life Violence Dataset"),
    "UCF-ARG (Multi-View)": Path(r"C:\Users\HPCLAB\Desktop\VLJEPA\capstone-data-root\cross camera data"),
    "TinyVIRAT-v2 (Complex)": Path(r"C:\Users\HPCLAB\Desktop\capstone-core datset\capstone_tinyvirat_v2\TinyVIRAT-v2\TinyVIRAT_V2"),
    "UCF-Crime (Anomalies)": Path(r"C:\Users\HPCLAB\Desktop\capstone-ucf dataset")
}

dataset_status = []
for name, p in dataset_paths.items():
    exists = p.exists()
    n_vids = len(list(p.rglob("*.mp4")) + list(p.rglob("*.avi"))) if exists else 0
    dataset_status.append({
        "Dataset Benchmark": name,
        "Directory Path": str(p),
        "Accessible": "✅ Verified" if exists else "❌ Missing",
        "Detected Videos": n_vids
    })

df_status = pd.DataFrame(dataset_status)
display(df_status)
print(f"\nTotal Real CCTV Videos Accessible: {sum(d['Detected Videos'] for d in dataset_status):,} clips across 4 benchmarks.")


## Step 6: Global Random Seed & Academic Reproducibility
Establish uniform random seeds across PyTorch, NumPy, and Python standard libraries to ensure 100% reproducible experiments.


In [ ]:
SEED = 42

def set_global_seed(seed: int = 42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"[Reproducibility] Global Seed locked to: {seed}")

set_global_seed(SEED)
print("Project setup complete. Ready to proceed to Notebook 01: CCTV Dataset Exploration.")
